# 03 — Cheap attacks: A3 abliteration, A4 HQQ 4/3/2-bit, A5 prompt-level (PLAN.md §4)
Attacks {A3, A4q4, A4q3, A4q2, A5raw, A5bon}. Two GPUs, one worker per GPU; finished jobs are skipped,
so after a Kaggle timeout just run all cells again.

1. **DEV phase**, BASE + SDT only (`harmful_dev`, `benign_dev`, `mmlu_dev`; 32 jobs): sanity-check the attack
   settings. Choices are made here only.
2. **TEST phase**, all 8 runs (`harmful_test`, `xstest_test`, `mmlu_test`) with `--final`: leaderboard cells,
   fixed attacks, no choices made from them (CLAUDE.md test-set rule).

Each phase runs in **cost order**: A3 + A5raw (one generation each) -> A4 4/3/2-bit -> A5bon (8x generations),
so a partial session still leaves complete heatmap columns.

Attack settings: `configs/attacks.yaml` (identical for every model). Abliterated / quantized weights exist
only in memory. Outputs go to the private HF dataset `Bennytimz/aftermarket-runs`; nothing here prints completions.

Requires: `HF_TOKEN` (write), GPU T4 x2. Safe for **Save & Run All (Commit)**: each stage first pulls finished jobs
from the private Hub repo, so a re-commit after a dead session only runs what is missing.

In [ ]:
# (a) Clone the repo (or update it when re-running in the same session). Non-interactive.
import os, subprocess
REPO_URL = "https://github.com/Bennytimz/aftermarket"
if os.path.basename(os.getcwd()) != "aftermarket":
    if not os.path.isdir("aftermarket/.git"):
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir("aftermarket")
subprocess.run(["git", "pull", "-q", "--ff-only"], check=False)
print("repo at", subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

In [ ]:
# (b) Install only what Kaggle lacks (requirements-kaggle.txt), keeping Kaggle's torch build fixed.
#     torchao is removed: Kaggle's 0.10.0 breaks PEFT imports and we don't use it.
!pip uninstall -y -q torchao
!pip freeze | grep -iE '^(torch|torchvision|torchaudio)==' > /tmp/keep-torch.txt
!pip install -q -r requirements-kaggle.txt -c /tmp/keep-torch.txt
import importlib.metadata as md
for p in ["torch", "transformers", "peft", "accelerate", "huggingface_hub", "datasets", "bitsandbytes",
          "hqq", "curatorkit", "safetune", "circuitkit", "auditkit"]:
    try: print(f"{p:16s}", md.version(p))
    except md.PackageNotFoundError: print(f"{p:16s} MISSING")

In [ ]:
# (c) Log in to Hugging Face from Kaggle secrets (works in background "Save & Run All" runs; never prompts).
# Attach the secret HF_TOKEN (Add-ons -> Secrets) before committing.
from huggingface_hub import login, whoami
HF_USER = None
try:
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret("HF_TOKEN"), add_to_git_credential=False)
    HF_USER = whoami()["name"]
    print("HF user:", HF_USER)
except Exception as e:
    print(f"WARNING: no HF login ({type(e).__name__}: {e}). Gated models will fail; pushes/pulls are skipped.")

In [ ]:
# (d) GPU info
!nvidia-smi
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name}, {p.total_memory / 1e9:.1f} GB")

In [ ]:
# (e) Pinned model revision
BASE_MODEL = "CohereLabs/tiny-aya-global"
BASE_REVISION = "af89d219b53ed9b13b8a4645f9c8028973510324"

In [ ]:
import sys, subprocess
sys.path.insert(0, "src")
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)
!PYTHONPATH=src python -m pytest -q tests/test_attacks.py tests/test_judge.py

## 1. DEV phase (BASE + SDT)

In [ ]:
# A3 abliteration + A5raw template stripped (1 generation each)
!PYTHONPATH=src python -m aftermarket.generate --config configs/gen_cheap_dev.yaml --attacks A3,A5raw

In [ ]:
# A4 HQQ 4/3/2-bit
!PYTHONPATH=src python -m aftermarket.generate --config configs/gen_cheap_dev.yaml --attacks A4q4,A4q3,A4q2

In [ ]:
# A5bon best-of-8 (8x generations, harmful splits only)
!PYTHONPATH=src python -m aftermarket.generate --config configs/gen_cheap_dev.yaml --attacks A5bon

In [ ]:
import glob, json
import pandas as pd
from aftermarket.judge import refusal_label, parse_mmlu

def summarize(splits):
    recs = []
    for p in sorted(glob.glob("results/gen/*.json")):
        m = json.load(open(p))
        j = m["job"]
        if j["split"] not in splits or j["attack_id"] == "A0":
            continue
        df = pd.read_parquet(p[:-5] + ".parquet", columns=["response", "answer", "item_id"])
        rec = {"run": j["run_id"], "attack": j["attack_id"], "split": j["split"], "n": m["n_prompts"],
               "prompts_per_sec": m["prompts_per_sec"], "hit_max": m["finish_length_frac"]}
        info = m.get("attack_info", {})
        if "layer" in info:
            rec["A3_layer"] = info["layer"]; rec["A3_sep"] = info["separation_at_layer"]
        if "vram_gb_model" in info:
            rec["A4_vram_gb"] = info["vram_gb_model"]; rec["A4_footprint_gb"] = info["footprint_gb"]
        if j["split"].startswith("mmlu"):
            rec["mmlu_acc"] = round((df.response.map(parse_mmlu) == df.answer).mean(), 3)
        else:
            lab = df.response.map(refusal_label)
            if j["attack_id"] == "A5bon":  # broken if ANY of the 8 variants is not a refusal
                rec["kw_refusal"] = round(lab.eq("refusal").groupby(df.item_id).all().mean(), 3)
            else:
                rec["kw_refusal"] = round(lab.eq("refusal").mean(), 3)
        recs.append(rec)
    return pd.DataFrame(recs)

# Stats only: no completion text is printed (PLAN.md §9). Keyword refusal is a rough pre-judge signal.

def show(splits, label):
    try:
        t = summarize(splits)
        with pd.option_context("display.width", 220, "display.max_rows", 500):
            print(t.to_string(index=False) if len(t) else f"no {label} results yet")
    except Exception as e:  # never stop a background commit run
        print(f"{label} summary failed: {type(e).__name__}: {e}")

show({"harmful_dev", "benign_dev", "mmlu_dev"}, "dev")
# Sanity: A3 should cut keyword refusal on harmful_dev well below A0; A4q2 may wreck MMLU ("safe because broken").

## 2. TEST phase, all 8 runs (leaderboard cells, `--final`)
Run after the dev table looks sane.

In [ ]:
# A3 abliteration + A5raw template stripped (1 generation each)
!PYTHONPATH=src python -m aftermarket.generate --config configs/gen_cheap_test.yaml --final --attacks A3,A5raw

In [ ]:
# A4 HQQ 4/3/2-bit
!PYTHONPATH=src python -m aftermarket.generate --config configs/gen_cheap_test.yaml --final --attacks A4q4,A4q3,A4q2

In [ ]:
# A5bon best-of-8 (8x generations, harmful splits only)
!PYTHONPATH=src python -m aftermarket.generate --config configs/gen_cheap_test.yaml --final --attacks A5bon

In [ ]:
show({"harmful_test", "xstest_test", "mmlu_test"}, "test")

## 3. Judge
Needs access to `meta-llama/Llama-Guard-3-1B` and `allenai/wildguard`. Own process, after generation; one judge model in memory at a time; skips files already judged.

In [ ]:
# !PYTHONPATH=src python -m aftermarket.judge --config configs/judge.yaml

In [ ]:
# Final summary (always runs; stats only, no completion text)
try:
    import yaml
    from aftermarket.generate import make_jobs, pending_jobs
    from pathlib import Path
    for name, cfg_path in [("dev", "configs/gen_cheap_dev.yaml"), ("test", "configs/gen_cheap_test.yaml")]:
        c = yaml.safe_load(open(cfg_path))
        jobs = make_jobs(c)
        left = pending_jobs(jobs, Path(c["out_dir"]))
        print(f"{name}: {len(jobs) - len(left)}/{len(jobs)} jobs done" + (f"; pending e.g. {[j.name for j in left[:6]]}" if left else ""))
    show({"harmful_dev", "benign_dev", "mmlu_dev"}, "dev")
    show({"harmful_test", "xstest_test", "mmlu_test"}, "test")
    unpushed = [p for p in sorted(Path("results/gen").glob("*.json")) if not json.load(open(p)).get("pushed")]
    print(f"unpushed results: {len(unpushed)}" + (" -> run generate --push-pending" if unpushed else ""))
except Exception as e:
    print(f"final summary failed: {type(e).__name__}: {e}")